# Materi 6: Aplikasi Klasifikasi Berita

Link : https://aplikasi-klasifikasi-berita-gtd5t2cad34pic3zgz5jyr.streamlit.app

## 1. Struktur dan urutan kerja

Aplikasi memiliki dua tahap. Tahap pertama adalah training yang membaca dataset berlabel dan menyimpan model. Tahap kedua adalah aplikasi Streamlit yang memuat model tersebut dan memproses URL baru.

Urutan: output/berita.csv → train_model.py → models/ → app/streamlit_app.py. File pipeline.py digunakan oleh training dan aplikasi agar preprocessing serta pembentukan vektor tetap sama.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("../aplikasi-klasifikasi-berita").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)
print("Project root tersedia:", PROJECT_ROOT.exists())

Project root: D:\Kuliah\Semester 7\Pencarian & Penambangan Web\Webmining\aplikasi-klasifikasi-berita
Project root tersedia: True


## 2. Training model

### Lokasi kode
aplikasi-klasifikasi-berita/train_model.py, baris 18–27

In [ ]:
def train(data_path: Path, project_root: Path) -> None:
    df = pd.read_csv(data_path)
    required = {"Isi Berita", "Label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Kolom dataset kurang: {sorted(missing)}")
    df = df[["Isi Berita", "Label"]].dropna()
    df["Label"] = df["Label"].astype(str).str.lower().str.strip()
    if set(df["Label"].unique()) != {"sport", "finance"}:
        raise ValueError(f"Label harus sport dan finance, ditemukan: {sorted(df['Label'].unique())}")

### Penjelasan

Fungsi train membaca CSV, memastikan kolom Isi Berita dan Label tersedia, menghapus baris kosong, lalu menyeragamkan label menjadi huruf kecil. Pemeriksaan label memastikan dataset yang digunakan memang sesuai dengan dua kelas aplikasi.

### Pembuktian

Cell berikut memeriksa dataset yang digunakan tanpa mengubah file sumber.

In [3]:
from pathlib import Path
import pandas as pd

data_path = Path("../output/berita.csv")
df = pd.read_csv(data_path)
print("Kolom:", list(df.columns))
print("Jumlah baris:", len(df))
print("Distribusi label:")
print(df["Label"].str.lower().str.strip().value_counts())

Kolom: ['ID', 'Isi Berita', 'Label']
Jumlah baris: 200
Distribusi label:
Label
sport      100
finance    100
Name: count, dtype: int64


Output pembuktian diharapkan menunjukkan 200 baris dengan distribusi 100 `sport` dan 100 `finance`.

### Lokasi kode
aplikasi-klasifikasi-berita/train_model.py, baris 28–30

In [ ]:
tokens = df["Isi Berita"].map(preprocess_text).tolist()
train_tokens, test_tokens, train_labels, test_labels = train_test_split(
    tokens,
    df["Label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["Label"].tolist(),
)
word2vec = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=42,
    epochs=100,
)

### Penjelasan

Artikel diubah menjadi token melalui preprocess_text. Data kemudian dibagi menjadi 80 persen training dan 20 persen testing. Parameter stratify menjaga proporsi label pada kedua bagian.

Word2Vec dilatih hanya menggunakan train_tokens. Parameter sg=1 menunjukkan arsitektur Skip-gram, vector_size=100 menentukan panjang vektor setiap kata, dan window=5 menentukan jarak konteks yang dipelajari.

### Pembuktian

Cell berikut memeriksa bahwa model Skip-gram memiliki vocabulary dan vektor dengan 100 dimensi.

In [5]:
from gensim.models import Word2Vec

model_path = Path("../aplikasi-klasifikasi-berita/models/word2vec_skipgram.model")
word2vec = Word2Vec.load(str(model_path))
print("Vector size:", word2vec.vector_size)
print("Jumlah kata dalam vocabulary:", len(word2vec.wv))
print("Contoh vektor kata:", word2vec.wv[list(word2vec.wv.key_to_index)[0]][:5])

Vector size: 100
Jumlah kata dalam vocabulary: 6361
Contoh vektor kata: [-0.46424922  0.6847447   0.49262947  0.00422061 -0.9292336 ]


Output menunjukkan Vector size: 100. Hal tersebut membuktikan bahwa setiap kata di dalam model direpresentasikan sebagai vektor 100 angka.

### Lokasi kode
aplikasi-klasifikasi-berita/train_model.py, baris 31–36

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(train_labels)
y_test = encoder.transform(test_labels)
X_train = np.array([document_vector(word2vec, row) for row in train_tokens])
X_test = np.array([document_vector(word2vec, row) for row in test_tokens])
classifier = GaussianNB().fit(X_train, y_train)
predictions = classifier.predict(X_test)

### Penjelasan

LabelEncoder mengubah label teks menjadi angka. Setiap artikel diubah menjadi satu document vector melalui rata-rata vektor kata. Gaussian Naive Bayes kemudian dilatih menggunakan document vector dan label training. Prediksi dibuat pada data testing yang tidak digunakan saat training.

### Pembuktian

Artefak model dan hasil evaluasi dibuat oleh script training. Cell berikut memeriksa model yang tersimpan dan melakukan prediksi sederhana menggunakan model tersebut.

In [7]:
import joblib
import numpy as np
from pipeline import document_vector, preprocess_text

root = Path("../aplikasi-klasifikasi-berita")
word2vec = Word2Vec.load(str(root / "models/word2vec_skipgram.model"))
classifier = joblib.load(root / "models/naive_bayes.pkl")
encoder = joblib.load(root / "models/label_encoder.pkl")

for text in [
    "Harga saham dan investasi pasar keuangan meningkat",
    "Tim sepak bola menang dalam pertandingan liga",
]:
    tokens = preprocess_text(text)
    vector = document_vector(word2vec, tokens).reshape(1, -1)
    prediction = encoder.inverse_transform(classifier.predict(vector))[0]
    print(repr(text), "->", prediction)

'Harga saham dan investasi pasar keuangan meningkat' -> finance
'Tim sepak bola menang dalam pertandingan liga' -> sport


### Lokasi kode
aplikasi-klasifikasi-berita/train_model.py, baris 37–48

In [ ]:
paths = model_paths(project_root)
paths["word2vec"].parent.mkdir(parents=True, exist_ok=True)
word2vec.save(str(paths["word2vec"]))
joblib.dump(classifier, paths["classifier"])
joblib.dump(encoder, paths["encoder"])
save_metadata(project_root, {"classes": list(encoder.classes_), "vector_size": 100, "window": 5, "sg": 1})

### Penjelasan

Word2Vec disimpan sebagai file .model, sedangkan classifier dan encoder disimpan sebagai file .pkl. Dengan cara ini aplikasi tidak perlu melatih ulang model setiap kali dijalankan.

### Pembuktian

Cell berikut memeriksa keberadaan artefak model dan ukuran filenya.

In [9]:
model_dir = Path("../aplikasi-klasifikasi-berita/models")
for name in ["word2vec_skipgram.model", "naive_bayes.pkl", "label_encoder.pkl", "model_metadata.pkl"]:
    path = model_dir / name
    print(name, "ADA" if path.exists() else "HILANG", end="")
    if path.exists():
        print(f" ({path.stat().st_size / 1024:.1f} KB)")
    else:
        print()

word2vec_skipgram.model ADA (5164.9 KB)
naive_bayes.pkl ADA (2.3 KB)
label_encoder.pkl ADA (0.4 KB)
model_metadata.pkl ADA (0.4 KB)


## 3. Fungsi bersama pada pipeline.py

### Lokasi kode
aplikasi-klasifikasi-berita/pipeline.py, baris 25–27

In [ ]:
def preprocess_text(text: str) -> list[str]:
    text = re.sub(r"[^a-z\s]", " ", str(text).lower())
    return [word for word in text.split() if word not in STOPWORDS and len(word) > 2]

### Penjelasan

Fungsi ini mengubah teks menjadi huruf kecil, mengganti karakter selain huruf dengan spasi, memisahkan teks menjadi token, menghapus stopword, dan membuang token dengan panjang maksimal dua karakter. Fungsi yang sama dipakai saat training dan saat aplikasi menerima artikel baru.

### Pembuktian

Contoh berikut menunjukkan perubahan teks sebelum dan sesudah preprocessing.

In [11]:
from pipeline import preprocess_text
contoh = "Harga saham naik 12,5% di bursa!"
print("Sebelum:", contoh)
print("Sesudah:", preprocess_text(contoh))

Sebelum: Harga saham naik 12,5% di bursa!
Sesudah: ['harga', 'saham', 'naik', 'bursa']


### Lokasi kode
aplikasi-klasifikasi-berita/pipeline.py, baris 30–34

In [ ]:
def document_vector(model: Word2Vec, tokens: Iterable[str]) -> np.ndarray:
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)

### Penjelasan dan pembuktian

Fungsi ini mengambil vektor setiap token yang terdapat di vocabulary Word2Vec, kemudian menghitung rata-ratanya. Hasilnya adalah satu vektor yang mewakili seluruh artikel. Jika tidak ada token yang dikenal, fungsi mengembalikan vektor nol dengan dimensi yang sama.

In [13]:
tokens = preprocess_text("Pemain mencetak gol pada pertandingan liga")
vector = document_vector(word2vec, tokens)
print("Jumlah token:", len(tokens))
print("Bentuk document vector:", vector.shape)
print("Lima nilai pertama:", vector[:5])

Jumlah token: 5
Bentuk document vector: (100,)
Lima nilai pertama: [ 0.1476594   0.30645898 -0.12146858 -0.773632    0.23589095]


## 4. Aplikasi Streamlit

### Lokasi kode
aplikasi-klasifikasi-berita/app/streamlit_app.py, baris 15–17 dan 19–27

In [ ]:
PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
from pipeline import document_vector, model_paths, preprocess_text

st.set_page_config(page_title="Klasifikasi Berita", page_icon="News", layout="centered")
st.title("Klasifikasi Berita")
st.caption("Word2Vec Skip-gram + Gaussian Naive Bayes")

### Penjelasan

PROJECT_ROOT menunjuk ke folder utama aplikasi. Folder tersebut dimasukkan ke sys.path agar pipeline.py dapat diimpor. Tiga baris Streamlit berikutnya mengatur judul halaman dan keterangan model.

### Lokasi kode
aplikasi-klasifikasi-berita/app/streamlit_app.py, baris 30–94

In [ ]:
def extract_article(url: str) -> tuple[str | None, str]:
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        return None, "URL tidak valid; gunakan URL yang diawali http:// atau https://"
    html = None
    fetch_error = None
    try:
        response = requests.get(
            url,
            headers={"User-Agent": USER_AGENT, "Accept-Language": "id-ID,id;q=0.9,en;q=0.8"},
            timeout=20,
            allow_redirects=True,
        )
        response.raise_for_status()
        html = response.text
    except requests.RequestException as error:
        fetch_error = f"request gagal: {error}"

### Penjelasan

Fungsi "extract_article" pertama-tama memvalidasi URL. Request dikirim dengan "User-Agent" yang menyerupai browser, batas waktu 20 detik, dan dukungan redirect. "response.raise_for_status()" membuat status HTTP seperti 403 atau 404 dianggap sebagai kegagalan. Jika berhasil, isi respons disimpan pada variabel "html".

### Pembuktian: respons yang diterima adalah HTML mentah

Pernyataan bahwa response.text berisi HTML mentah dibuktikan dengan memeriksa tipe data, potongan awal respons, keberadaan tag HTML, dan perbedaan panjang antara respons mentah dan teks artikel yang sudah diekstrak. Cell ini tidak menjalankan Streamlit; cell ini menguji operasi HTTP yang sama dengan fungsi aplikasi.

In [16]:
import requests
import trafilatura

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/131.0.0.0 Safari/537.36"
)

uji_url = "https://lambeturah.co.id/post/kepala-bgn-bongkar-yayasan-minta-setoran-per-ompreng-dari-mitra-dapur-mbg"
response = requests.get(
    uji_url,
    headers={"User-Agent": USER_AGENT, "Accept-Language": "id-ID,id;q=0.9,en;q=0.8"},
    timeout=20,
    allow_redirects=True,
)
response.raise_for_status()
html = response.text
article = trafilatura.extract(html, include_comments=False, include_tables=False, favor_precision=False)

print("Status HTTP:", response.status_code)
print("Tipe response.text:", type(html).__name__)
print("Panjang HTML mentah:", len(html))
print("Awal respons:", repr(html[:180]))
print("Memuat tag HTML:", "<html" in html.lower() or "<div" in html.lower())
print("Panjang hasil ekstraksi:", len(article or ""))
print("Awal hasil ekstraksi:", repr((article or "")[:180]))

Status HTTP: 200
Tipe response.text: str
Panjang HTML mentah: 66066
Awal respons: '<!DOCTYPE html><html lang="id"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" href="https://lambeturah.co.id/'
Memuat tag HTML: True
Panjang hasil ekstraksi: 2522
Awal hasil ekstraksi: 'Lambeturah.co.id - Kepala Badan Gizi Nasional (BGN) Sudaryono mengungkap adanya yayasan yang menaungi dapur Satuan Pelayanan Pemenuhan Gizi (SPPG) yang diduga meminta setoran kepad'


Hasil yang diharapkan menunjukkan "response.text" bertipe "str", bagian awalnya memuat markup seperti "<html", "<head", atau "<div", sedangkan hasil "trafilatura.extract" berupa teks artikel tanpa tag HTML. Inilah bukti bahwa aplikasi menerima HTML mentah terlebih dahulu, kemudian mengubahnya menjadi teks yang dapat diproses model.

### Lokasi kode
aplikasi-klasifikasi-berita/app/streamlit_app.py, baris 51–94

In [ ]:
if html:
    article = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=False,
        favor_precision=False,
    )
    if article and len(article.strip()) >= 120:
        return article.strip(), "trafilatura (requests)"

soup = BeautifulSoup(html, "html.parser")
for tag in soup(["script", "style", "nav", "header", "footer", "aside", "form", "noscript"]):
    tag.decompose()

candidates = soup.select(", ".join(selectors))
if candidates:
    text = max((node.get_text(" ", strip=True) for node in candidates), key=len)
    if len(text) >= 120:
        return text, "BeautifulSoup (article container)"

### Penjelasan dan pembuktian

Aplikasi mencoba Trafilatura terlebih dahulu. Jika hasilnya tidak mencapai 120 karakter, aplikasi menggunakan BeautifulSoup sebagai fallback. Elemen yang biasanya bukan isi berita, seperti `script`, `style`, `nav`, dan `footer`, dihapus. Setelah itu aplikasi memilih container artikel dengan teks terpanjang.

Pada URL Lambeturah yang diuji, proses berhasil menghasilkan teks artikel. Metode ekstraksi yang berhasil dikembalikan melalui `extraction_method` dan ditampilkan pada aplikasi.

### Lokasi kode
aplikasi-klasifikasi-berita/app/streamlit_app.py, baris 97–104 dan 136–153

In [ ]:
@st.cache_resource
def load_models():
    paths = model_paths(PROJECT_ROOT)
    required = [paths["word2vec"], paths["classifier"], paths["encoder"]]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Model belum tersedia. Jalankan train_model.py terlebih dahulu.")
    return Word2Vec.load(str(paths["word2vec"])), joblib.load(paths["classifier"]), joblib.load(paths["encoder"])

tokens = preprocess_text(article)
vector = document_vector(word2vec, tokens).reshape(1, -1)
prediction = classifier.predict(vector)
probabilities = classifier.predict_proba(vector)[0]
label = encoder.inverse_transform(prediction)[0]

### Penjelasan

@st.cache_resource membuat model dimuat sekali dan digunakan kembali ketika Streamlit melakukan rerun. Artikel yang telah diekstrak diproses dengan fungsi yang sama seperti training, diubah menjadi document vector, lalu diberikan kepada Gaussian Naive Bayes. Hasil numerik dikembalikan menjadi label sport atau finance menggunakan LabelEncoder.

Aplikasi saat ini adalah classifier biner. Artikel politik, kesehatan, hiburan, atau kategori lain tetap akan dipaksa masuk ke salah satu dari dua kelas tersebut karena model belum memiliki label other.

## 6. Kesimpulan

Aplikasi menggunakan dataset berita berlabel untuk melatih Word2Vec Skip-gram dan Gaussian Naive Bayes. Saat digunakan, aplikasi menerima URL, memperoleh HTML mentah melalui request HTTP, mengekstrak teks artikel, menjalankan preprocessing yang sama dengan tahap training, lalu menghasilkan prediksi.

Pembuktian di notebook menunjukkan bahwa respons web memang berupa string HTML mentah sebelum diproses oleh Trafilatura atau BeautifulSoup. Pembuktian lainnya memeriksa dataset, vocabulary Word2Vec, document vector, artefak model, serta prediksi dari model yang sudah disimpan.